<a href="https://colab.research.google.com/github/fandre01/Machine-Leaning/blob/main/Copy_of_starter_bank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
campaign = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bank.csv')

# import polars as pl
# campaign = pl.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bank.csv', schema_overrides={'nr.employed': pl.Float64})

In [3]:
campaign.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [4]:
campaign.isna().sum()

,0
age,0
job,0
marital,0
education,0
default,0
housing,0
loan,0
contact,0
month,0
day_of_week,0


In [5]:
unknow_counts= (campaign == "unknown").sum()
unknow_counts

,0
age,0
job,294
marital,69
education,1535
default,7725
housing,894
loan,894
contact,0
month,0
day_of_week,0


In [6]:
#Converts All text columns to numbers automatically
campaign_encoded = pd.get_dummies(campaign, drop_first=True)
campaign_encoded.head()

,age,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,job_blue-collar,...,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,y_yes
0,56,1,999,0,1.1,93.994,-36.4,4.857,5191.0,False,...,False,False,False,True,False,False,False,True,False,False
1,57,1,999,0,1.1,93.994,-36.4,4.857,5191.0,False,...,False,False,False,True,False,False,False,True,False,False
2,37,1,999,0,1.1,93.994,-36.4,4.857,5191.0,False,...,False,False,False,True,False,False,False,True,False,False
3,40,1,999,0,1.1,93.994,-36.4,4.857,5191.0,False,...,False,False,False,True,False,False,False,True,False,False
4,56,1,999,0,1.1,93.994,-36.4,4.857,5191.0,False,...,False,False,False,True,False,False,False,True,False,False


In [7]:
# convert my target to 1/ 0
#campaign['y'].map({"yes" : 0, 'no': 0})

In [8]:
# Convert the target as an integer
campaign_encoded = campaign_encoded.astype(int)

In [9]:
from sklearn.model_selection import train_test_split

# Separate features (X) from the target (y)
X = campaign_encoded.drop('y_yes', axis=1)
y = campaign_encoded['y_yes']

In [10]:
# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
# Check the size
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(29655, 52)
(7414, 52)
(29655,)
(7414,)


In [12]:
# Let's build the decision tree model
from sklearn.tree import DecisionTreeClassifier
# Create the model
dt_model = DecisionTreeClassifier(random_state=42)
# Let Train the model
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [13]:
# Check for accurancy
# I use class_weight='balanced' to tell the model even more people say no I most care about the say, please focus on yes.
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
dt_model = DecisionTreeClassifier(random_state=42,
                                   class_weight='balanced')
dt_model.fit(X_train, y_train)

predictions = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print("F1 Score:", f1_score(y_test, predictions))

Accuracy: 0.8393579714054491
F1 Score: 0.3312745648512072


In [14]:
# Let build the RandomForestClassifier model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

rf_model = RandomForestClassifier(n_estimators=100,
                                   random_state=42,
                                   class_weight='balanced')
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, rf_predictions))
print("F1 Score:", f1_score(y_test, rf_predictions))

Accuracy: 0.8835985972484489
F1 Score: 0.3407181054239878


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Create model
cl_model = RandomForestClassifier(
    n_estimators=100, # use 100 trees
    random_state=42, # same results every run
    class_weight='balanced' # pay more attention to "yes" customers
)

# Train model
cl_model.fit(X_train, y_train)

#Instead of yes/no, ask the model:
#"How CONFIDENT are you this person will say yes?"
#Get probability scores instead of direct predictions.
y_prob = cl_model.predict_proba(X_test)[:, 1]

# Lower threshold from 0.5 to 0.2
y_pred = (y_prob >= 0.2).astype(int)

# Evaluate model
print(classification_report(y_test, y_pred))




              precision    recall  f1-score   support

           0       0.94      0.88      0.91      6528
           1       0.39      0.56      0.46       886

    accuracy                           0.84      7414
   macro avg       0.66      0.72      0.68      7414
weighted avg       0.87      0.84      0.85      7414



In [52]:
#Load new data
new_data = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bank_holdout_test.csv')

#Encode same way as training data
new_data_encoded = pd.get_dummies(new_data, drop_first=True)
new_data_encoded = new_data_encoded.astype(int)

#Match columns with our trained model
new_data_encoded = new_data_encoded.reindex(columns=X_train.columns, fill_value=0)

#Make predictions
predictions = cl_model.predict(new_data_encoded)

#Save to CSV
my_predictions = pd.DataFrame(predictions, columns=['predictions'])
my_predictions.to_csv("Fabrice-predictions.csv", index=False)

#Check results
print("Total predictions:", len(my_predictions))
print("Predicted yes (1):", my_predictions['predictions'].sum())
print("Predicted no (0):", (my_predictions['predictions'] == 0).sum())

Total predictions: 4119
Predicted yes (1): 230
Predicted no (0): 3889


In [50]:
check = pd.read_csv('predictions.csv')
print(check.shape)        # should be (4119, 1)
print(check.columns)      # should be ['predictions']
print(check['predictions'].unique())  # should be [0, 1] only


(4119, 1)
Index(['predictions'], dtype='object')
[0 1]
